# Apuntes Extensos: Despliegue de Modelos de PyTorch (Model Deployment)

Este notebook constituye una guía completa, teórica y ejecutable para llevar modelos de Deep Learning desde el entorno de experimentación en Jupyter a producción. En el contexto de un proyecto de investigación o tesis, el despliegue es el puente entre las métricas estáticas del conjunto de prueba y el valor práctico del modelo ante datos del mundo real.

## Objetivos de este Documento:
1. **Comprender el Despliegue (Deployment):** Concepto, justificación e importancia del ciclo MLOps.
2. **Tomar Decisiones de Arquitectura:** Analizar los pros y contras del cómputo en el extremo (*Edge/On-device*) frente a la nube (*Cloud*), y de la inferencia en tiempo real (*Online*) frente a lotes (*Batch*).
3. **Preparar Modelos Heterogéneos:** Implementar *Transfer Learning* utilizando un extractor convolucional (EfficientNet-B2) y uno basado en atención (Vision Transformer).
4. **Evaluar el Compromiso Velocidad-Precisión (*Speed-Accuracy Trade-off*):** Comparar empíricamente latencias, tamaño en disco y exactitud en CPU.
5. **Prototipar con Gradio:** Crear interfaces gráficas web dinámicas y compartibles.
6. **Empaquetar para Producción:** Crear los archivos mínimos necesarios (`app.py`, `model.py`, `requirements.txt`).
7. **Desplegar en Hugging Face Spaces:** Hospedar la aplicación en la web de forma permanente.
8. **Escalar a FoodVision Big:** Entrenar el modelo convolucional con las 101 clases de Food101 y empaquetar su despliegue.


## 0. Configuración del Entorno

Comenzamos importando las dependencias necesarias de PyTorch, Torchvision y utilidades. Asimismo, nos aseguramos de clonar o importar los scripts modulares de `going_modular` creados en fases previas, y el módulo de funciones auxiliares `helper_functions.py` para graficar curvas de pérdida y descargar datos.


In [ ]:
# Aseguramos compatibilidad de APIs
import torch
import torchvision
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

# Instalamos torchinfo para ver resúmenes del modelo si no está disponible
try:
    from torchinfo import summary
except:
    print("[INFO] torchinfo no encontrado, instalando...")
    !pip install -q torchinfo
    from torchinfo import summary

# Importamos y preparamos scripts modulares
try:
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves
except:
    print("[INFO] Descargando scripts de going_modular y helper_functions desde GitHub...")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de cómputo seleccionado: {device}")


## 1. Descarga y Preparación de Datos

Para evaluar las dos arquitecturas de manera homogénea y bajo las mismas condiciones (comparación equitativa o *apples-to-apples*), utilizaremos el dataset **pizza_steak_sushi_20_percent**, que contiene el 20% del total de las imágenes de estas tres clases seleccionadas de la base de datos Food101.


In [ ]:
# Descarga del dataset del 20%
data_20_percent_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
    destination="pizza_steak_sushi_20_percent"
)

# Definición de rutas
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"
print(f"Ruta de entrenamiento: {train_dir}")
print(f"Ruta de prueba (test): {test_dir}")


## 2. Marco Experimental: Compromiso Velocidad-Precisión

El objetivo del despliegue en producción es lograr una excelente experiencia del usuario final. En sistemas interactivos (como enfocar comida con la cámara de un móvil), esto se traduce en dos métricas clave:
1. **Precisión (Exactitud):** $\ge 95\%$ de exactitud en clasificación multiclase.
2. **Velocidad (Latencia):** Tiempo de inferencia inferior a $\sim 0.033\text{ segundos}$ por imagen (equivalente a procesar a $\ge 30\text{ FPS}$ en tiempo real).

Para intentar cumplir estas métricas, entrenaremos y compararemos dos modelos:
- **EfficientNet-B2:** Un modelo convolucional (CNN) diseñado mediante búsqueda de arquitectura neuronal (NAS) para ser altamente eficiente.
- **Vision Transformer (ViT-B/16):** Un modelo basado en mecanismos de autoatención que divide las imágenes en parches de $16 \times 16$ y tiene mayor capacidad de aprendizaje pero una cantidad significativamente mayor de parámetros.


## 3. Extractor de Características 1: EfficientNet-B2

EfficientNet escala uniformemente la profundidad, ancho y resolución de la red mediante un coeficiente compuesto. Para usarlo como extractor de características:
1. Importamos la red con sus pesos preentrenados (`weights=EfficientNet_B2_Weights.DEFAULT`).
2. Congelamos todas las capas del extractor base.
3. Reemplazamos la capa de clasificación final (`classifier`) para mapear el vector latente de 1408 características hacia nuestras 3 clases objetivo, incorporando un Dropout del $30\%$.


In [ ]:
def create_effnetb2_model(num_classes: int = 3, seed: int = 42):
    """Instancia un extractor de características de EfficientNet-B2 y sus transformaciones."""
    # 1. Cargamos pesos y transformaciones por defecto
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.efficientnet_b2(weights=weights)

    # 2. Congelamos la red base
    for param in model.parameters():
        param.requires_grad = False

    # 3. Personalizamos el clasificador final
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )
    return model, transforms

# Crear el modelo y los DataLoaders correspondientes
effnetb2, effnetb2_transforms = create_effnetb2_model(num_classes=3, seed=42)
train_dataloader_effnetb2, test_dataloader_effnetb2, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=effnetb2_transforms,
    batch_size=32
)


In [ ]:
# Configuración del optimizador y función de pérdida
optimizer_effnet = torch.optim.Adam(params=effnetb2.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Entrenamos el modelo por 10 épocas
set_seeds()
effnetb2_results = engine.train(
    model=effnetb2,
    train_dataloader=train_dataloader_effnetb2,
    test_dataloader=test_dataloader_effnetb2,
    epochs=10,
    optimizer=optimizer_effnet,
    loss_fn=loss_fn,
    device=device
)

# Visualizar curvas de pérdida
plot_loss_curves(effnetb2_results)


## 4. Extractor de Características 2: Vision Transformer (ViT-B/16)

El Vision Transformer (ViT) divide las imágenes en parches y aplica un Encoder de Transformer estándar. La codificación espacial se añade sumando embeddings de posición. El cabezal de clasificación se alimenta del `class_token` especial.

Seguiremos el mismo proceso: cargar el modelo preentrenado `vit_b_16`, congelar el codificador base y modificar la capa final (`heads`) para clasificar nuestras 3 categorías.


In [ ]:
def create_vit_model(num_classes: int = 3, seed: int = 42):
    """Instancia un extractor de características de ViT-B/16 y sus transformaciones."""
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.vit_b_16(weights=weights)

    # Congelamos las capas base
    for param in model.parameters():
        param.requires_grad = False

    # Reemplazamos la capa de clasificación lineal de la cabeza
    torch.manual_seed(seed)
    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes)
    )
    return model, transforms

# Crear el modelo y los DataLoaders de ViT
vit, vit_transforms = create_vit_model(num_classes=3, seed=42)
train_dataloader_vit, test_dataloader_vit, _ = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=vit_transforms,
    batch_size=32
)


In [ ]:
# Configuración del entrenamiento para ViT
optimizer_vit = torch.optim.Adam(params=vit.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Entrenamos el modelo por 10 épocas
set_seeds()
vit_results = engine.train(
    model=vit,
    train_dataloader=train_dataloader_vit,
    test_dataloader=test_dataloader_vit,
    epochs=10,
    optimizer=optimizer_vit,
    loss_fn=loss_fn,
    device=device
)

# Visualizar curvas de pérdida
plot_loss_curves(vit_results)


## 5. Inferencia en CPU y Temporización Individual

Para el despliegue es crucial medir el tiempo de predicción de una sola imagen a la vez (no por lotes). Dado que en producción el servidor web usará la CPU por defecto, realizaremos la inferencia forzando `device='cpu'` y guardaremos la metadata e historial de tiempos de respuesta.


In [ ]:
import pathlib
from PIL import Image
from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import List, Dict, Tuple

def pred_and_store(paths: List[pathlib.Path],
                   model: torch.nn.Module,
                   transform: torchvision.transforms,
                   class_names: List[str],
                   device: str = "cpu") -> List[Dict]:
    """Genera predicciones en CPU imagen por imagen y registra los tiempos de inferencia."""
    pred_list = []
    for path in tqdm(paths):
        pred_dict = {}
        pred_dict["image_path"] = path
        class_name = path.parent.stem
        pred_dict["class_name"] = class_name
        
        # Iniciar cronómetro
        start_time = timer()
        img = Image.open(path)
        
        # Transformar imagen y añadir batch dimension
        transformed_image = transform(img).unsqueeze(0).to(device)
        model.to(device)
        model.eval()
        
        with torch.inference_mode():
            pred_logit = model(transformed_image)
            pred_prob = torch.softmax(pred_logit, dim=1)
            pred_label = torch.argmax(pred_prob, dim=1)
            pred_class = class_names[pred_label.cpu()]
            
            # Almacenar predicción, probabilidad y tiempo
            pred_dict["pred_prob"] = round(pred_prob.max().cpu().item(), 4)
            pred_dict["pred_class"] = pred_class
            end_time = timer()
            pred_dict["time_for_pred"] = round(end_time - start_time, 4)
            
        pred_dict["correct"] = class_name == pred_class
        pred_list.append(pred_dict)
    return pred_list

# Obtener rutas de imágenes de prueba
test_data_paths = list(Path(test_dir).glob("*/*.jpg"))
print(f"Cantidad de imágenes de prueba: {len(test_data_paths)}")


In [ ]:
import pandas as pd

# 1. Inferencia con EffNetB2
effnetb2_test_pred_dicts = pred_and_store(paths=test_data_paths,
                                          model=effnetb2,
                                          transform=effnetb2_transforms,
                                          class_names=class_names,
                                          device="cpu")

effnetb2_test_pred_df = pd.DataFrame(effnetb2_test_pred_dicts)
effnetb2_average_time_per_pred = round(effnetb2_test_pred_df.time_for_pred.mean(), 4)
print(f"Tiempo medio EffNetB2 por predicción en CPU: {effnetb2_average_time_per_pred} segundos")

# 2. Inferencia con ViT
vit_test_pred_dicts = pred_and_store(paths=test_data_paths,
                                     model=vit,
                                     transform=vit_transforms,
                                     class_names=class_names,
                                     device="cpu")

vit_test_pred_df = pd.DataFrame(vit_test_pred_dicts)
vit_average_time_per_pred = round(vit_test_pred_df.time_for_pred.mean(), 4)
print(f"Tiempo medio ViT por predicción en CPU: {vit_average_time_per_pred} segundos")


## 6. Comparación Sistemática y Visualización de Ratios

Confrontamos las métricas obtenidas por ambos modelos. Construiremos un DataFrame con los resultados y graficaremos el compromiso velocidad-exactitud.


In [ ]:
# Recopilación de estadísticas
effnetb2_stats = {
    "test_loss": effnetb2_results["test_loss"][-1],
    "test_acc": effnetb2_results["test_acc"][-1],
    "number_of_parameters": sum(p.numel() for p in effnetb2.parameters()),
    "model_size (MB)": 29,
    "time_per_pred_cpu": effnetb2_average_time_per_pred,
    "model": "EffNetB2"
}

vit_stats = {
    "test_loss": vit_results["test_loss"][-1],
    "test_acc": vit_results["test_acc"][-1],
    "number_of_parameters": sum(p.numel() for p in vit.parameters()),
    "model_size (MB)": 327,
    "time_per_pred_cpu": vit_average_time_per_pred,
    "model": "ViT"
}

# Crear DataFrame
df = pd.DataFrame([effnetb2_stats, vit_stats])
df["test_acc"] = round(df["test_acc"] * 100, 2)
print("Tabla Comparativa de Modelos:")
print(df)

# Ratios de ViT con respecto a EffNetB2
ratios = pd.DataFrame(df.set_index("model").loc["ViT"] / df.set_index("model").loc["EffNetB2"],
                      columns=["ViT to EffNetB2 Ratio"]).T
print("\nRatios (ViT / EffNetB2):")
print(ratios)


In [ ]:
# Graficamos el trade-off
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(data=df,
                     x="time_per_pred_cpu",
                     y="test_acc",
                     c=["blue", "orange"],
                     s="model_size (MB)", # Tamaño del marcador proporcional al peso en MB
                     alpha=0.6)

ax.set_title("Compromiso Velocidad vs Precisión de Inferencia (CPU)", fontsize=16)
ax.set_xlabel("Tiempo de predicción por imagen (segundos)", fontsize=12)
ax.set_ylabel("Exactitud en Test (%)", fontsize=12)
ax.grid(True)

for index, row in df.iterrows():
    ax.annotate(text=row["model"],
                xy=(row["time_per_pred_cpu"] + 0.001, row["test_acc"] + 0.05),
                size=11)

handles, labels = scatter.legend_elements(prop="sizes", alpha=0.5)
ax.legend(handles, labels, loc="lower right", title="Tamaño del Modelo (MB)")
plt.show()


## 7. Creación de una Interfaz Gráfica con Gradio

Gradio permite empaquetar nuestro modelo de visión computacional bajo la lógica `Inputs -> ML Model / Function -> Outputs`. Definiremos una función `predict()` para procesar una sola imagen de tipo PIL y desplegaremos la interfaz gráfica localmente.


In [ ]:
try:
    import gradio as gr
except:
    !pip install -q gradio
    import gradio as gr

import random
effnetb2.to("cpu")

def predict(img) -> Tuple[Dict, float]:
    """Mapea la imagen PIL de entrada hacia predicciones de clase y tiempo de cómputo."""
    start_time = timer()
    img_tensor = effnetb2_transforms(img).unsqueeze(0).to("cpu")
    
    effnetb2.eval()
    with torch.inference_mode():
        pred_probs = torch.softmax(effnetb2(img_tensor), dim=1)
    
    # Estructura del diccionario para Gradio (Label)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    pred_time = round(timer() - start_time, 4)
    return pred_labels_and_probs, pred_time

# Crear lista de ejemplos
example_list = [[str(filepath)] for filepath in random.sample(test_data_paths, k=3)]

# Lanzamiento de la interfaz
demo = gr.Interface(fn=predict,
                    inputs=gr.Image(type="pil"),
                    outputs=[gr.Label(num_top_classes=3, label="Predictions"),
                             gr.Number(label="Prediction time (s)")],
                    examples=example_list,
                    title="FoodVision Mini 🍕🥩🍣",
                    description="Un extractor de características EfficientNetB2 para clasificar pizza, steak o sushi.",
                    article="Creado para apuntes académicos sobre PyTorch.")

# demo.launch(share=True) # Descomentar para lanzar interfaz interactiva en local y generar link compartido


## 8. Empaquetado y Estructura del Proyecto Local

Para garantizar un despliegue exitoso en Hugging Face Spaces, estructuraremos la aplicación local en un directorio modular, generando programáticamente los archivos `model.py`, `app.py` y `requirements.txt`.


In [ ]:
import shutil
import os
from pathlib import Path

# Directorio de trabajo local para la app
foodvision_mini_demo_path = Path("demos/foodvision_mini/")
if foodvision_mini_demo_path.exists():
    shutil.rmtree(foodvision_mini_demo_path)
foodvision_mini_demo_path.mkdir(parents=True, exist_ok=True)

# Copiar ejemplos al directorio
foodvision_mini_examples_path = foodvision_mini_demo_path / "examples"
foodvision_mini_examples_path.mkdir(parents=True, exist_ok=True)

for example in random.sample(test_data_paths, k=3):
    shutil.copy2(src=example, dst=foodvision_mini_examples_path / example.name)

# Guardar el estado de pesos del modelo
model_save_path = Path("models/09_pretrained_effnetb2_feature_extractor_pizza_steak_sushi_20_percent.pth")
if not model_save_path.parent.exists():
    model_save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(effnetb2.state_dict(), model_save_path)
shutil.copy2(src=model_save_path, dst=foodvision_mini_demo_path / model_save_path.name)


In [ ]:
%%writefile demos/foodvision_mini/model.py
import torch
import torchvision
from torch import nn

def create_effnetb2_model(num_classes:int=3, seed:int=42):
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.efficientnet_b2(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )
    return model, transforms


In [ ]:
%%writefile demos/foodvision_mini/app.py
import gradio as gr
import os
import torch
from model import create_effnetb2_model
from timeit import default_timer as timer
from typing import Tuple, Dict

class_names = ["pizza", "steak", "sushi"]

effnetb2, effnetb2_transforms = create_effnetb2_model(num_classes=3)
effnetb2.load_state_dict(
    torch.load(
        f="09_pretrained_effnetb2_feature_extractor_pizza_steak_sushi_20_percent.pth",
        map_location=torch.device("cpu")
    )
)

def predict(img) -> Tuple[Dict, float]:
    start_time = timer()
    img = effnetb2_transforms(img).unsqueeze(0)
    effnetb2.eval()
    with torch.inference_mode():
        pred_probs = torch.softmax(effnetb2(img), dim=1)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    pred_time = round(timer() - start_time, 4)
    return pred_labels_and_probs, pred_time

title = "FoodVision Mini 🍕🥩🍣"
description = "An EfficientNetB2 feature extractor computer vision model to classify images of food."
article = "Created as academic notes."

example_list = [["examples/" + example] for example in os.listdir("examples")]
demo = gr.Interface(fn=predict,
                    inputs=gr.Image(type="pil"),
                    outputs=[gr.Label(num_top_classes=3, label="Predictions"),
                             gr.Number(label="Prediction time (s)")],
                    examples=example_list,
                    title=title,
                    description=description,
                    article=article)
demo.launch()


In [ ]:
%%writefile demos/foodvision_mini/requirements.txt
torch==1.12.0
torchvision==0.13.0
gradio==3.1.4


## 9. Despliegue en Hugging Face Spaces

Para subir este directorio empaquetado a Hugging Face Spaces:
1. **Crea una cuenta gratuita** en [Hugging Face](https://huggingface.co/).
2. **Crea un nuevo Space** en la plataforma, seleccionando `Gradio` como SDK y de acceso público.
3. **Clona el repositorio** en tu entorno local mediante Git.
4. **Copia los archivos** generados en tu directorio `demos/foodvision_mini/` al interior del repositorio Git clonado.
5. **Configura Git LFS (Large File Storage):** Dado que el archivo de pesos del modelo (`.pth`) pesa más de 10 MB, Git LFS es obligatorio para subir el modelo de PyTorch. Ejecuta en terminal:
   * `git lfs install`
   * `git lfs track "*.pth"` (añadirá el archivo a `.gitattributes`)
6. **Añade, confirma y sube los cambios** con:
   * `git add .`
   * `git commit -m "Subida inicial del clasificador FoodVision Mini"`
   * `git push`


## 10. Escalado: Creación de FoodVision Big (101 clases)

Para pasar del clasificador de 3 clases al dataset completo de Food101 (101 clases de comida y 100,000 imágenes), entrenaremos nuestro modelo EfficientNet-B2 en un subconjunto del 20% del total del dataset (15,150 imágenes en train y 5,050 en test) para optimizar el tiempo de ejecución en la máquina.

Utilizaremos **Label Smoothing (0.1)** en CrossEntropyLoss para evitar que el modelo se sobreajuste con excesiva confianza en una única etiqueta, ayudando a generalizar mejor.


In [ ]:
from torchvision import datasets

# Transformaciones aumentadas para el entrenamiento de 101 clases
food101_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),
    effnetb2_transforms
])

# Obtener el dataset completo
train_data = datasets.Food101(root="data", split="train", transform=food101_train_transforms, download=True)
test_data = datasets.Food101(root="data", split="test", transform=effnetb2_transforms, download=True)
food101_class_names = train_data.classes


In [ ]:
def split_dataset(dataset: torchvision.datasets, split_size: float = 0.2, seed: int = 42):
    """Divide el dataset en proporciones aleatorias reproducibles."""
    length_1 = int(len(dataset) * split_size)
    length_2 = len(dataset) - length_1
    split_1, split_2 = torch.utils.data.random_split(
        dataset,
        lengths=[length_1, length_2],
        generator=torch.manual_seed(seed)
    )
    return split_1, split_2

# Obtenemos la partición del 20%
train_data_food101_20_percent, _ = split_dataset(dataset=train_data, split_size=0.2)
test_data_food101_20_percent, _ = split_dataset(dataset=test_data, split_size=0.2)

# Creamos DataLoaders
train_dataloader_food101_20_percent = torch.utils.data.DataLoader(
    train_data_food101_20_percent, batch_size=32, shuffle=True, num_workers=2
)
test_dataloader_food101_20_percent = torch.utils.data.DataLoader(
    test_data_food101_20_percent, batch_size=32, shuffle=False, num_workers=2
)


In [ ]:
# Inicializamos y entrenamos el modelo para 101 clases
effnetb2_food101, _ = create_effnetb2_model(num_classes=101, seed=42)
optimizer_big = torch.optim.Adam(params=effnetb2_food101.parameters(), lr=1e-3)
loss_fn_big = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

# El entrenamiento del modelo de 101 clases se limita a 5 épocas por cómputo
set_seeds()
effnetb2_food101_results = engine.train(
    model=effnetb2_food101,
    train_dataloader=train_dataloader_food101_20_percent,
    test_dataloader=test_dataloader_food101_20_percent,
    optimizer=optimizer_big,
    loss_fn=loss_fn_big,
    epochs=5,
    device=device
)


## 11. Empaquetado y Despliegue de FoodVision Big

Creamos la carpeta del demo `demos/foodvision_big/` escribiendo los archivos `class_names.txt`, `model.py`, `app.py` y `requirements.txt` de manera análoga al modelo mini.


In [ ]:
# Crear directorio
foodvision_big_demo_path = Path("demos/foodvision_big/")
foodvision_big_demo_path.mkdir(parents=True, exist_ok=True)
foodvision_big_examples_path = foodvision_big_demo_path / "examples"
foodvision_big_examples_path.mkdir(parents=True, exist_ok=True)

# Guardar pesos del modelo FoodVision Big
torch.save(effnetb2_food101.state_dict(), foodvision_big_demo_path / "09_pretrained_effnetb2_feature_extractor_food101_20_percent.pth")

# Escribir los nombres de las clases en class_names.txt
with open(foodvision_big_demo_path / "class_names.txt", "w") as f:
    f.write("\n".join(food101_class_names))


In [ ]:
%%writefile demos/foodvision_big/model.py
import torch
import torchvision
from torch import nn

def create_effnetb2_model(num_classes:int=3, seed:int=42):
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.efficientnet_b2(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )
    return model, transforms


In [ ]:
%%writefile demos/foodvision_big/app.py
import gradio as gr
import os
import torch
from model import create_effnetb2_model
from timeit import default_timer as timer
from typing import Tuple, Dict

with open("class_names.txt", "r") as f:
    class_names = [food_name.strip() for food_name in f.readlines()]

effnetb2, effnetb2_transforms = create_effnetb2_model(num_classes=101)
effnetb2.load_state_dict(
    torch.load(
        f="09_pretrained_effnetb2_feature_extractor_food101_20_percent.pth",
        map_location=torch.device("cpu")
    )
)

def predict(img) -> Tuple[Dict, float]:
    start_time = timer()
    img = effnetb2_transforms(img).unsqueeze(0)
    effnetb2.eval()
    with torch.inference_mode():
        pred_probs = torch.softmax(effnetb2(img), dim=1)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    pred_time = round(timer() - start_time, 4)
    return pred_labels_and_probs, pred_time

title = "FoodVision Big 🍔👁"
description = "An EfficientNetB2 computer vision model to classify images of food into 101 different classes."
article = "Created as academic notes."

# Se creará la lista de ejemplos si se agregan imágenes a la carpeta de examples
example_list = []
if os.path.exists("examples") and len(os.listdir("examples")) > 0:
    example_list = [["examples/" + example] for example in os.listdir("examples")]

demo = gr.Interface(fn=predict,
                    inputs=gr.Image(type="pil"),
                    outputs=[gr.Label(num_top_classes=5, label="Predictions"),
                             gr.Number(label="Prediction time (s)")],
                    examples=example_list if len(example_list) > 0 else None,
                    title=title,
                    description=description,
                    article=article)
demo.launch()


In [ ]:
%%writefile demos/foodvision_big/requirements.txt
torch==1.12.0
torchvision==0.13.0
gradio==3.1.4


In [ ]:
# Comprimimos los archivos del demo FoodVision Big para su descarga
!cd demos/foodvision_big && zip -r ../foodvision_big.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"
